# 01_map_lens_ids_to_families

**Objective:** Map each Lens patent ID to its DOCDB family ID by joining the patent master's family members against the publication-number indexes of the EPO and USPTO PaECTER embedding files.

**Inputs:** `patent_master.parquet`; EPO and USPTO PaECTER embedding parquets.

**Outputs:** `lens_to_familyid.parquet` with columns `lens_id`, `docdb_family_id`, `source`, `matched_publno`.

In [1]:
# Imports
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

In [2]:
# Paths and configuration
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
EPO        = RAW / "embeddings" / "EPO_PaECTER_embeddings.parquet"
USPTO      = RAW / "embeddings" / "USPTO_PaECTER_embeddings.parquet"
MASTER     = PROC / "patent_master.parquet"
OUTPUT     = PROC / "lens_to_familyid.parquet"

LIST_COLS = ["applicants", "applicant_countries", "cpc_codes", "ipc_codes",
             "family_members", "bwd_cit_lens_ids", "bwd_cit_docids"]

In [3]:
# Coerce array-like values into plain Python lists
def to_pylist(x):
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (list, tuple)):
        return list(x)
    return []

In [4]:
# Build publno -> row-index lookup from a PaECTER parquet
def load_publno_lookup(parquet_path: Path, label: str) -> dict:
    print(f"\nLoading {label}: {parquet_path.name} ...")
    df = pd.read_parquet(parquet_path, columns=["publno"])
    print(f"  rows: {len(df):,}")
    n_null = df["publno"].isna().sum()
    if n_null:
        print(f"  null publnos: {n_null:,} (drop)")
        df = df.dropna(subset=["publno"])
    n_dupes = df["publno"].duplicated().sum()
    if n_dupes:
        print(f"  duplicates: {n_dupes:,} (keep first)")
        df = df[~df["publno"].duplicated(keep="first")]
    lookup = dict(zip(df["publno"].values, df.index.values))
    print(f"  lookup size: {len(lookup):,}")
    return lookup

In [5]:
# Generate publno candidate variants from family member strings
def gen_publno_variants(family_members):
    out, seen = [], set()
    for fm in family_members or []:
        parts = fm.split("|")
        if len(parts) < 2:
            continue
        juris, docnum = parts[0], parts[1]
        kind = parts[2] if len(parts) > 2 else ""
        if not juris or not docnum:
            continue
        candidates = []
        if kind:
            candidates.append(f"{juris}{docnum}{kind}")
        candidates.append(f"{juris}{docnum}")
        if docnum.startswith("0"):
            stripped = docnum.lstrip("0")
            if kind:
                candidates.append(f"{juris}{stripped}{kind}")
            candidates.append(f"{juris}{stripped}")
        if docnum.isdigit() and len(docnum) < 7:
            padded = docnum.zfill(7)
            if kind:
                candidates.append(f"{juris}{padded}{kind}")
            candidates.append(f"{juris}{padded}")
        for c in candidates:
            if c not in seen:
                seen.add(c)
                out.append(c)
    return out

In [6]:
# Match family members to a family id, EPO first then USPTO
def match_to_family_id(family_members, epo_lookup, uspto_lookup):
    candidates = gen_publno_variants(family_members)
    for c in candidates:
        if c in epo_lookup:
            return (int(epo_lookup[c]), "EPO", c)
    for c in candidates:
        if c in uspto_lookup:
            return (int(uspto_lookup[c]), "USPTO", c)
    return (None, None, None)

In [7]:
# Main pipeline: load inputs, match, summarize, and save
def main():
    for p in [EPO, USPTO, MASTER]:
        if not p.exists():
            raise FileNotFoundError(f"Missing input: {p}")
        print(f"OK: {p.name}  ({p.stat().st_size / 1e9:.2f} GB)")

    master = pd.read_parquet(MASTER)
    for col in LIST_COLS:
        if col in master.columns:
            master[col] = master[col].apply(to_pylist)
    n_with_fam = (master["family_members"].str.len() > 0).sum()
    print(f"\nMaster: {len(master):,} rows, {n_with_fam:,} with family members")

    epo_lookup   = load_publno_lookup(EPO,   "EPO")
    uspto_lookup = load_publno_lookup(USPTO, "USPTO")

    print(f"\nMatching {len(master):,} patents ...")
    results = [
        match_to_family_id(fm, epo_lookup, uspto_lookup)
        for fm in tqdm(master["family_members"], total=len(master))
    ]

    mapping = pd.DataFrame({
        "lens_id":         master["lens_id"].values,
        "docdb_family_id": [r[0] for r in results],
        "source":          [r[1] for r in results],
        "matched_publno":  [r[2] for r in results],
    })
    mapping["docdb_family_id"] = mapping["docdb_family_id"].astype("Int64")

    n = len(mapping)
    n_matched = mapping["docdb_family_id"].notna().sum()
    print(f"\nMatched: {n_matched:,} / {n:,}  ({100*n_matched/n:.1f}%)")
    print("By source:")
    print(mapping["source"].value_counts(dropna=False).to_string())

    mapping.to_parquet(OUTPUT, compression="zstd")
    print(f"\nWrote: {OUTPUT}  ({OUTPUT.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Entry point
if __name__ == "__main__":
    main()